In [ ]:
import math
import os
from tempfile import TemporaryDirectory
from typing import Tuple
import matplotlib.pyplot as plt

import torch
from torch import nn, Tensor
from torch.nn import TransformerEncoder, TransformerEncoderLayer
from torch.utils.data import DataLoader

import sys
sys.path.append('..')
from model.utils import MLP, PositionalEncoding
from model import HIMTransfomerNet as net
from model import TransformerHIM, OptimHIM
from dataset import SeqTeleopDataset
from model.riccati import dare as DARE

%load_ext autoreload
%autoreload 2

In [ ]:
M = 2
B = 1
D = 10
H = 128
O = 4

In [ ]:
dataset = SeqTeleopDataset('../data/lqr_optimal/R2e1', index=[0, 1])
dataloader = DataLoader(dataset, batch_size=B)

In [ ]:
data = next(iter(dataloader))

In [ ]:
states, actions, states_next = data

In [ ]:
states.shape, actions.shape, states_next.shape

# Grount Truth

In [ ]:
pos, goal = torch.chunk(states, 2, dim=-1)
pos_next, goal_next = torch.chunk(states_next, 2, dim=-1)

In [ ]:
model = TransformerHIM(D, O)

In [ ]:
A = torch.eye(2)
B = torch.eye(2) * 0.001
Q = torch.eye(2) * 5
R = torch.eye(2) * 2
# W = torch.tensor([-5e-2, 0.0])

In [ ]:
seq_len = states.shape[1]
B = B.expand(*states.shape[:2], -1, -1)
# W = W.expand(*states.shape[:2], -1)

B.requires_grad = True
# W.requires_grad = True

In [ ]:
dare = DARE()
P = dare(A, B, Q, R)
K = model.lqr_K(P, A, B, Q, R)

In [ ]:
# K[0, 0].numpy(force=True), sigma[0, 0].numpy(force=True)

In [ ]:
x = pos - goal
u_H_star = - torch.matmul(K, x.unsqueeze(-1)).squeeze(-1)
# u_H_star += torch.sign(x) * W
u_H = actions #- torch.sign(x) * W

In [ ]:
u_H_star, u_H

In [ ]:
x_next_star = model.dynamics(A, B, x, u_H_star)
x_next_pred = model.dynamics(A, B, x, u_H)

In [ ]:
Q_u_H = model.value_Q(P, Q, R, x, u_H, x_next_pred)   # (B, T)
Q_u_H_star = model.value_Q(P, Q, R, x, u_H_star, x_next_star)  # (B, T)

In [ ]:
likelihoods = model.likelihood_u_H(P, B, R, Q_u_H - Q_u_H_star)
loss = - likelihoods.sum(dim=-1).mean()
loss.backward()

In [ ]:
vals = likelihoods.detach().cpu().flatten().numpy()

plt.figure(figsize=(8, 4))
plt.hist(vals, bins=40, edgecolor='black')
plt.title("Histogram of likelihoods")
plt.xlabel("Likelihood value")
plt.ylabel("Frequency")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
B.grad#, W.grad

In [ ]:
B.grad.sum(axis=[1])

In [ ]:
deltas = x_next_star - pos_next
deltas = deltas.squeeze(0).numpy(force=True)

import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

ax1.plot(deltas[:, 0], label='Delta X')
ax1.set_ylabel('Delta Value')
ax1.set_title('X State Prediction Error')
ax1.legend()
ax1.grid(True)

ax2.plot(deltas[:, 1], label='Delta Y')
ax2.set_xlabel('Time Step')
ax2.set_ylabel('Delta Value')
ax2.set_title('Y State Prediction Error')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

# Test

In [ ]:
model = net(
    d_model=D,
    d_out=O,
    d_hid=H,
    batch_first=True
)

In [ ]:
d = torch.cat([states, actions, states_next], dim=-1)  # (B, T, D)

In [ ]:
embedded = model(d)

In [ ]:
d.shape, embedded.shape

In [ ]:
b, w = torch.chunk(embedded, 2, dim=-1)
N = b.shape[-1]
b_0 = torch.eye(N, device=b.device, dtype=b.dtype)
b_0 = b_0.expand(B, 1, N, N)
b = torch.diag_embed(b)
b = torch.cat([b_0, b], dim=1)

In [ ]:
dare_net = DARE()

In [ ]:
a = torch.eye(M).unsqueeze(0).unsqueeze(1)
q = torch.eye(M).unsqueeze(0).unsqueeze(1)
r = torch.eye(M).unsqueeze(0).unsqueeze(1)

In [ ]:
a.shape, b.shape

In [ ]:
p = dare_net.forward(a, b, q, r)

In [ ]:
def lqr_K(P: Tensor, A: Tensor, B: Tensor, Q: Tensor, R: Tensor) -> Tensor:
    """
    Solve the discrete-time LQR controller for a batch of systems.

    Args:
        P: (..., n, n) solution to the discrete-time Riccati equation
        A: (..., n, n) state transition matrices
        B: (..., n, m) control input matrices
        Q: (..., n, n) state cost matrices
        R: (..., m, m) control cost matrices
    Returns:
        K: (..., m, n) optimal gain matrices
    """

    BT_P = torch.matmul(B.transpose(-1, -2), P)  # (..., m, n)
    BT_P_B = torch.matmul(BT_P, B)  # (..., m, m)
    R_plus = R + BT_P_B  # (..., m, m)
    R_plus_inv = torch.linalg.inv(R_plus)  # (..., m, m)

    BT_P_A = torch.matmul(BT_P, A)  # (..., m, n)

    K = torch.matmul(R_plus_inv, BT_P_A)  # (..., m, n)

    return K

In [ ]:
def q_h(P: Tensor, Q: Tensor, R: Tensor, x: Tensor, u: Tensor, x_next: Tensor) -> Tensor:
    """
    Compute the Q-function for a batch of systems.

    Args:
        P: (..., n, n) solution to the discrete-time Riccati equation
        Q: (..., n, n) state cost matrices
        R: (..., m, m) control cost matrices
        x: (..., n) states
        u: (..., m) controls
        x_next: (..., n) next states
    Returns:
        q: (...) Q-function values
    """
    
    x_Q = torch.matmul(x.unsqueeze(-2), Q)  # (..., 1, n)
    x_Q_x = torch.matmul(x_Q, x.unsqueeze(-1)).squeeze(-1)  # (...,)

    u_R = torch.matmul(u.unsqueeze(-2), R)  # (..., 1, m)
    u_R_u = torch.matmul(u_R, u.unsqueeze(-1)).squeeze(-1)  # (...,)

    x_next_P = torch.matmul(x_next.unsqueeze(-2), P)  # (..., 1, n)
    x_next_P_x_next = torch.matmul(x_next_P, x_next.unsqueeze(-1)).squeeze(-1)  # (...,)

    q = - x_Q_x - u_R_u - x_next_P_x_next  # (...,)

    return q.squeeze(-1)

In [ ]:
k = lqr_K(p, a, b, q, r)

In [ ]:
pos, goal = torch.chunk(states, 2, dim=-1)
pos_next, _ = torch.chunk(states_next, 2, dim=-1)

In [ ]:
u_h = actions - torch.sign(pos - goal) * w
u_h_star = -torch.matmul(k[:, :-1], pos.unsqueeze(-1)).squeeze(-1)

In [ ]:
u_h.shape, u_h_star.shape

In [ ]:
q_h_u = q_h(p[:, 1:], q, r, pos, u_h, pos_next)
q_h_u_star = q_h(p[:, 1:], q, r, pos, u_h_star, pos_next)

In [ ]:
q_h_u.shape

In [ ]:
def u_h_prob(P: Tensor, B: Tensor, R: Tensor, delta_Q: Tensor):
    """
    Compute the probability of human control input under the optimal policy.

    Args:
        P: (..., n, n) solution to the discrete-time Riccati equation
        B: (..., n, m) control input matrices
        R: (..., m, m) control cost matrices
        delta_Q: (...) difference in Q-function values
    Returns:
        prob: (...) probabilities of human control inputs
    """

    BT_P = torch.matmul(B.transpose(-1, -2), P)  # (..., m, n)
    BT_P_B = torch.matmul(BT_P, B)  # (..., m, m)
    
    H = 2 * R + 2 * BT_P_B  # (..., m, m)

    det_H = torch.det(H)  # (...)

    m_h = B.shape[-1] / 2
    coff = - m_h * torch.exp(delta_Q)

    prob = torch.sqrt(det_H) * (2 * math.pi) ** coff

    return prob

In [ ]:
prob = u_h_prob(p[:, 1:], b[:, 1:], r, q_h_u - q_h_u_star)

In [ ]:
torch.log(prob).sum(dim=-1)